In [1]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
import os
import winsound

# Monte Carlo Simulations of our System

## Code 

### Create the system

In [2]:
def wrap_angle(x):
    '''
    Wrap angle between -pi/6 and pi/6
    Inputs:
        x: angle
    Outputs:
        wrapped angle
    '''

    return (x + np.pi/6) % (np.pi/3) - np.pi/6

def theta_flatten(theta, indices):
    '''
    Flatten theta array
    Inputs:
        theta: 2D
        indices
    Outputs:
        theta_flat: 1D
    '''

    return np.array([theta[i, j] for (i, j) in indices])

class Lattice:
    '''
    Lattice class to hold lattice properties
    Attributes:
        L: size
        indices: list of (i,j) coordinates
        voisins: list of neighbors for each (i,j)
    '''

    def __init__(self, L, rho=1.0):
        self.L = L
        self.rho = rho
        self.indices = np.array([(i, j) for i in range(L) for j in range(L)])

        self.voisins = [[] for _ in range(L*L)]
        for idx, (i, j) in enumerate(self.indices):
            if i % 2 == 0:
                candidats = [
                (i + 1, j),
                (i - 1, j),
                (i, j + 1),
                (i, j - 1),
                (i + 1, j - 1),
                (i - 1, j - 1)
            ]
            else:
                candidats = [
                (i + 1, j),
                (i - 1, j),
                (i, j + 1),
                (i, j - 1),
                (i + 1, j + 1),
                (i - 1, j + 1)
            ]
            v_valid = [(ni, nj) for (ni, nj) in candidats if 0 <= ni < L and 0 <= nj < L]
            self.voisins[idx] = v_valid
        self.theta = np.random.uniform(-np.pi/6, np.pi/6, size=(L, L))

        aire = self.L * self.L / self.rho
        self.distance = np.sqrt(aire) / self.L
        self.d_x = self.distance
        self.d_y = self.distance * np.sqrt(3) / 2
        xs, ys = [], []
        for (i, j) in self.indices:
            x = j * self.d_x + (i % 2) * (self.d_x / 2)
            y = i * self.d_y
            xs.append(x)
            ys.append(y)
        self.xs = np.array(xs)
        self.ys = np.array(ys)

        self.dX = self.xs[:, None] - self.xs[None, :]
        self.dY = self.ys[:, None] - self.ys[None, :]

        self.r = np.sqrt(self.dX**2 + self.dY**2)
        r_max = np.max(self.r)

        self.N_bins = int(self.L)
        self.bins = np.linspace(0, r_max, self.N_bins + 1)
        self.r_vals = 0.5 * (self.bins[:-1] + self.bins[1:])



    def get_xy_positions(self, i, j):
        '''
        Get (x,y) of the (i,j) point 
        Inputs:
            i, j
        Outputs:
            x, y
        '''

        x = j * self.d_x + (i % 2) * (self.d_x / 2)
        y = i * self.d_y
        return x, y
    
    def hamiltonien(self, epsilon, gamma, A):
        '''
        Compute the Hamiltonian of the system
        Inputs:
            epsilon, gamma, A: parameters
        Outputs:
            Hamiltonian
        '''

        H_0 = (epsilon/self.rho) * np.sum(theta_flatten(self.theta, self.indices)**2)
        H_int = 0.0
        for idx, (i, j) in enumerate(self.indices):
            t_ij = self.theta[i, j]
            for (ni, nj) in self.voisins[idx]:
                dtheta = np.abs(wrap_angle(t_ij - self.theta[ni, nj]))
                H_int += dtheta * (A - np.log(max(dtheta, 1e-12)))
        H_int = (gamma / 3 * np.sqrt(self.rho)) * H_int
        H_int /= 2
        return H_0 + H_int

    def dhamiltonien(self, new_theta, idx, epsilon, gamma, A):
        '''
        Compute the change in Hamiltonian when one theta is changed
        Inputs:
            new_theta
            idx: index of the changed theta in indices
            epsilon, gamma, A: parameters
        Outputs:
            Change in Hamiltonian
        '''

        i, j = self.indices[idx]
        old_theta = self.theta[i, j]
        dH_0 = (epsilon/self.rho) * (new_theta**2 - old_theta**2)
        dH_int = 0.0
        for (ni, nj) in self.voisins[idx]:
            dtheta_old = np.abs(wrap_angle(old_theta - self.theta[ni, nj]))
            dtheta_new = np.abs(wrap_angle(new_theta - self.theta[ni, nj]))
            dH_int += dtheta_new * (A - np.log(max(dtheta_new, 1e-12))) - dtheta_old * (A - np.log(max(dtheta_old, 1e-12)))
        dH_int = (gamma / 3 * np.sqrt(self.rho)) * dH_int
        return dH_0 + dH_int

### Monte Carlo Simulation

In [3]:
class Metropolis:
    def __init__(self, delta_init=0.2, target_acceptance=0.5, tune_interval=100_000, growth_factor=1.05, shrink_factor=0.95, min_delta=1e-3, max_delta=np.pi/6):
        self.delta = delta_init
        self.target_acceptance = target_acceptance
        self.tune_interval = tune_interval
        self.growth_factor = growth_factor
        self.shrink_factor = shrink_factor
        self.min_delta = min_delta
        self.max_delta = max_delta
        self.attempts = 0
        self.acceptances = 0

    def metropolis_step(self, lattice, epsilon, gamma, A, beta, tune=True):
        '''
        Perform one Metropolis step with adaptive delta
        Inputs:
            lattice: Lattice object
            epsilon, gamma, A: parameters
            beta
        '''
        N = len(lattice.indices)
        idx = np.random.randint(N)
        i, j = lattice.indices[idx]
        new_theta = wrap_angle(lattice.theta[i, j] + np.random.uniform(-self.delta, self.delta))
        delta_H = lattice.dhamiltonien(new_theta, idx, epsilon, gamma, A)

        self.attempts += 1
        if delta_H < 0 or np.random.rand() < np.exp(-beta * delta_H):
            lattice.theta[i, j] = new_theta
            self.acceptances += 1

        if tune and self.attempts % self.tune_interval == 0:
            acceptance_rate = self.acceptances / max(1, self.attempts)
            if acceptance_rate < self.target_acceptance - 0.05:
                self.delta = max(self.min_delta, self.delta * self.shrink_factor)
            elif acceptance_rate > self.target_acceptance + 0.05:
                self.delta = min(self.max_delta, self.delta * self.growth_factor)
            self.attempts = 0
            self.acceptances = 0

    def overrelaxation_step(self, lattice, epsilon, gamma, A, beta):
        '''
        Perform one overrelaxation step
        Inputs:
            lattice: Lattice object
            epsilon, gamma, A: parameters
            beta
        '''

        for idx, (i, j) in enumerate(lattice.indices):
            sum_sin6 = 0.0
            sum_cos6 = 0.0
            for (ni, nj) in lattice.voisins[idx]:
                sum_cos6 += np.cos(6 * lattice.theta[ni, nj])
                sum_sin6 += np.sin(6 * lattice.theta[ni, nj])
            
            if sum_cos6 == 0 and sum_sin6 == 0:
                continue

            phi6 = np.arctan2(sum_sin6, sum_cos6)
            theta_old = lattice.theta[i, j]
            theta_new = wrap_angle((2 * phi6 / 6) - theta_old)
            delta_H = lattice.dhamiltonien(theta_new, idx, epsilon, gamma, A)
            if delta_H < 0 or np.random.rand() < np.exp(-beta * delta_H):
                lattice.theta[i, j] = theta_new

    
class Simulation:
    def __init__(self, L, T, epsilon, gamma, A, rho=1.0, n_thermal=50_000, n_steps=10_000, overrelax_interval=1000, measure_interval=1000, tune_interval=100_000, tune_delta=True, delta_init=0.2, target_acceptance=0.5, growth_factor=1.05, shrink_factor=0.95, min_delta=1e-3, max_delta=np.pi/6):
        self.L = L
        self.T = T
        self.epsilon = epsilon
        self.gamma = gamma
        self.A = A
        self.beta = 1.0 / T
        self.rho = rho
        self.n_thermal = n_thermal
        self.n_steps = n_steps
        self.overrelax_interval = overrelax_interval
        self.measure_interval = measure_interval
        self.lattice = Lattice(L, rho)
        self.metropolis = Metropolis(delta_init=delta_init, target_acceptance=target_acceptance, tune_interval=tune_interval, growth_factor=growth_factor, shrink_factor=shrink_factor, min_delta=min_delta, max_delta=max_delta)
        self.tune_delta = tune_delta
        self.energies = []
        self.r = None
        self.G = None
        self.G_err = None

    def compute_correlation(self):
        '''
        Compute the correlation function of the lattice
        Outputs:
            r : distance
            G : correlation function
        '''
        
        psi6 = np.exp(1j * 6 * self.lattice.theta)
        psi6_flat = theta_flatten(psi6, self.lattice.indices)

        C = np.outer(psi6_flat, np.conj(psi6_flat)).real

        r = self.lattice.r

        iu, ju = np.triu_indices(self.L * self.L, k=1)
        r_ij = r[iu, ju]
        C_ij = C[iu, ju]

        n_bins = self.lattice.N_bins
        bins = self.lattice.bins
        r_vals = self.lattice.r_vals
        G_vals = np.zeros(n_bins)
        counts = np.zeros(n_bins, dtype=int)

        which_bins = np.digitize(r_ij, bins) - 1
        mask = (which_bins >= 0) & (which_bins < n_bins)
        np.add.at(G_vals, which_bins[mask], C_ij[mask])
        np.add.at(counts, which_bins[mask], 1)

        r_vals = np.insert(r_vals, 0, 0.0)
        G_vals = np.insert(G_vals, 0, self.L * self.L)
        counts = np.insert(counts, 0, self.L * self.L)

        G_vals = np.where(counts > 0, G_vals / counts, 0.0)
        G_vals = G_vals / G_vals[0]

        return r_vals, G_vals

    def save_results(self, path):
        '''
        Save simulation results to files
        Inputs:
            path: Directory path to save the results
        '''
        
        os.makedirs(path, exist_ok=True)

        theta = theta_flatten(self.lattice.theta, self.lattice.indices)
        x_positions = self.lattice.xs
        y_positions = self.lattice.ys

        header = "x,y,theta"
        data = np.column_stack((x_positions, y_positions, theta))
        np.savetxt(f"{path}/Final_Configuration_T{self.T:.2e}_L{self.L}.csv", data, delimiter=",", header=header, comments='')

        header = "step,energy_per_site"
        data = np.column_stack((np.arange(len(self.energies)) * (self.n_thermal // 10), self.energies))
        np.savetxt(f"{path}/Energy_Evolution_T{self.T:.2e}_L{self.L}.csv", data, delimiter=",", header=header, comments='')

        header = "r,G,G_error"
        data = np.column_stack((self.r, self.G, self.G_err))
        np.savetxt(f"{path}/Correlation_Function_T{self.T:.2e}_L{self.L}.csv", data, delimiter=",", header=header, comments='')

    def one_step(self, step, tune=True):
        '''
        Perform one simulation step (Metropolis + Overrelaxation)
        '''
        
        if self.overrelax_interval > 0 and step % self.overrelax_interval == 0:
            self.metropolis.overrelaxation_step(self.lattice, self.epsilon, self.gamma, self.A, self.beta)
        else:
            self.metropolis.metropolis_step(self.lattice, self.epsilon, self.gamma, self.A, self.beta, tune=tune)

    def run(self, path):
        '''
        Run the Monte Carlo simulation
        Inputs:
            path: Directory path to save the results
        '''
        
        display(Markdown(f"## Running Simulation at T={self.T:.2e}, L={self.L}"))

        for step in range(self.n_thermal + 1):
            self.one_step(step, tune=self.tune_delta)
            if step % (self.n_thermal // 10) == 0:
                energy = self.lattice.hamiltonien(self.epsilon, self.gamma, self.A) / (self.L * self.L)
                self.energies.append(energy)
        
        rs, Gs = [], []

        for step in range(self.n_steps + 1):
            self.one_step(step, tune=False)
            if self.measure_interval > 0 and step % self.measure_interval == 0:
                r, G = self.compute_correlation()
                rs.append(r)
                Gs.append(G)

        self.r = np.mean(rs, axis=0)
        self.G = np.mean(Gs, axis=0)
        self.G_err = np.std(Gs, axis=0) / np.sqrt(len(Gs))

        rate = (self.metropolis.acceptances /
                max(1, self.metropolis.attempts))

        display(Markdown(f"### Final acceptance rate: {rate:.3f}, Final delta: {self.metropolis.delta:.4f} rad"))
        self.save_results(path)

### Plot results

In [4]:
def plot_results(path, L, T):
    '''
    Plot the results of the simulation
    Inputs:
        path: path to the directory where the results are saved
        L: lattice size
        T: temperature
    '''
    
    theta_data = np.loadtxt(f"{path}/Final_Configuration_T{T:.2e}_L{L}.csv", delimiter=",", skiprows=1)
    x_positions = theta_data[:, 0]
    y_positions = theta_data[:, 1]
    theta = theta_data[:, 2]

    energy_data = np.loadtxt(f"{path}/Energy_Evolution_T{T:.2e}_L{L}.csv", delimiter=",", skiprows=1)
    steps = energy_data[:, 0]
    energies = energy_data[:, 1]

    corr_data = np.loadtxt(f"{path}/Correlation_Function_T{T:.2e}_L{L}.csv", delimiter=",", skiprows=1)
    r = corr_data[:, 0]
    G = corr_data[:, 1]
    G_err = corr_data[:, 2]

    plt.figure(figsize=(8, 6))
    plt.scatter(x_positions, y_positions, c=theta, cmap='hsv', s=40, marker='h')
    plt.colorbar(label=r'$\theta$ (rad)')
    plt.title(rf'Hexagonal Lattice L={L}, $T={T:.2e}$')
    plt.xlabel(r'$x$')
    plt.ylabel(r'$y$')
    plt.savefig(f"{path}/Final_Configuration_T{T:.2e}_L{L}.pdf")
    plt.close()

    plt.figure(figsize=(8, 6))
    plt.plot(steps, energies)
    plt.title(rf'Energy per Spin at $T={T:.2e}$, $L={L}$')
    plt.xlabel('Monte Carlo Steps')
    plt.ylabel(r'$E/N$')
    plt.grid()
    plt.savefig(f"{path}/Energy_Evolution_T{T:.2e}_L{L}.pdf")
    plt.close()

    r_min = r[0]
    r_max = 2 * r[-1] / 3
    G_threshold = 1e-4
    mask = (r > r_min) & (r < r_max) & (G > G_threshold)
    coef_power_decay, cov_power_decay = np.polyfit(np.log(r[mask]), np.log(G[mask]), 1, cov=True)
    a_power, b_power = coef_power_decay
    eta6 = -a_power
    eta6_err = np.sqrt(cov_power_decay[0, 0])
    A_power = np.exp(b_power)

    coef_expo_decay, cov_expo_decay = np.polyfit(r[mask], np.log(G[mask]), 1, cov=True)
    a_expo, b_expo = coef_expo_decay
    xi6 = -1 / a_expo
    A = np.exp(b_expo)
    xi6_err = xi6**2 * np.sqrt(cov_expo_decay[0, 0])

    plt.figure(figsize=(8, 6))
    plt.errorbar(r, np.abs(G), yerr=G_err, fmt='-^', label='Simulation Data')
    plt.plot(r[1:], A_power * r[1:]**(-eta6), linestyle='--', label=rf'Power-law fit: $\eta_6={eta6:.3f} \pm {eta6_err:.3f}$')
    plt.plot(r, A * np.exp(-r/xi6), linestyle=':', label=rf'Exponential fit: $\xi_6={xi6:.2f} \pm {xi6_err:.2f}$')
    plt.title(rf'Correlation Function at $T={T:.2e}$, $L={L}$')
    plt.xlabel(r'$r$')
    plt.ylabel(r'$G_6(r)$')
    plt.ylim(0, 1.1)
    plt.legend()
    plt.grid()
    plt.savefig(f"{path}/Correlation_Function_T{T:.2e}_L{L}.pdf")
    plt.close()

    plt.figure(figsize=(8, 6))
    plt.errorbar(r, np.abs(G), yerr=G_err, fmt='-^', label='Simulation Data')
    plt.plot(r[1:], A_power * r[1:]**(-eta6), linestyle='--', label=rf'Power-law fit: $\eta_6={eta6:.3f} \pm {eta6_err:.3f}$')
    plt.plot(r, A * np.exp(-r/xi6), linestyle=':', label=rf'Exponential fit: $\xi_6={xi6:.2f} \pm {xi6_err:.2f}$')
    plt.xscale('log')
    plt.yscale('log')
    plt.title(rf'Correlation Function at $T={T:.2e}$, $L={L}$ (Log-Log Scale)')
    plt.xlabel(r'$r$')
    plt.ylabel(r'$G_6(r)$')
    plt.legend()
    plt.grid()
    plt.savefig(f"{path}/Correlation_Function_T{T:.2e}_L{L}_loglog.pdf")
    plt.close()

## Simulation 

In [5]:
epsilon = 0
gamma = 2
rho = 1.0
A = 1.0

n_thermal=5*10**5
n_steps=10**5
measure_interval=n_steps // 10
tune_interval=n_thermal // 100
overrelax_interval=n_thermal // 200

T = [10**(-5), 0.01, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1, 0.15, 0.16, 0.17, 0.18, 0.19, 0.2]
L=32
for temp in T:
    path = f"Epsilon{epsilon}Gamma{gamma}/L{L}/T{temp:.2e}"
    sim = Simulation(L=L, T=temp, epsilon=epsilon, gamma=gamma, A=A, rho=rho, n_thermal=n_thermal, n_steps=n_steps, overrelax_interval=overrelax_interval, measure_interval=measure_interval, tune_interval=tune_interval, tune_delta=True)
    sim.run(path=path)

winsound.Beep(500, 2000)

## Running Simulation at T=1.00e-05, L=32

### Final acceptance rate: 0.008, Final delta: 0.0012 rad

## Running Simulation at T=1.00e-02, L=32

### Final acceptance rate: 0.485, Final delta: 0.0026 rad

## Running Simulation at T=5.00e-02, L=32

...

### Final acceptance rate: 0.484, Final delta: 0.0715 rad

## Running Simulation at T=1.90e-01, L=32

### Final acceptance rate: 0.484, Final delta: 0.0792 rad

## Running Simulation at T=2.00e-01, L=32

### Final acceptance rate: 0.492, Final delta: 0.0834 rad

In [6]:
epsilon = 0
gamma = 2
T = [10**(-5), 0.01, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1, 0.15, 0.16, 0.17, 0.18, 0.19, 0.2]
L=32
for temp in T:
    path = f"Epsilon{epsilon}Gamma{gamma}/L{L}/T{temp:.2e}"
    plot_results(path, L, temp)

In [ ]:
### test correlation